<a href="https://colab.research.google.com/github/hinamehboobcs/Industry-Project/blob/main/GA_Dynamic1_Approved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [83]:
import pandas as pd

# =========================
# LOAD DATA
# =========================
carers = pd.read_csv("instance2_carers.csv")
visits = pd.read_csv("instance2_visits.csv")

# =========================
# TIME CONVERSION
# =========================
def time_to_minutes(t):
    if pd.isna(t):
        return None
    h, m = map(int, t.split(":"))
    return h * 60 + m

# =========================
# PRIORITY MAPPING
# =========================
priority_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

# =========================
# STRING → SET CONVERTER
# =========================
def to_set(x):
    if pd.isna(x):
        return set()
    return set([i.strip() for i in str(x).split(",")])

# =========================
# CLEAN CARERS
# =========================
carers_clean = []

for _, row in carers.iterrows():
    carers_clean.append({
        "CarerID": row["CarerID"],
        "Date": row["Date"],
        "Day": row["DayOfWeek"],
        "Latitude": row["Latitude"],
        "Longitude": row["Longitude"],
        "PreferredLat": row["PreferredAreaLatitude"],
        "PreferredLon": row["PreferredAreaLongitude"],
        "MaxDistance": row["MaxTravelDistanceMiles"],
        "Experience": row["ExperienceInYears"],
        "Skills": to_set(row["Skills"]),
        "Gender": row["Gender"],
        "Languages": to_set(row["Languages"]),
        "Vehicle": row["VehicleType"],
        "WorkingDays": to_set(row["WorkingDays"]),
        "OffDays": to_set(row["OffDays"]),
        "ShiftStart": time_to_minutes(row["ShiftStart"]),
        "ShiftEnd": time_to_minutes(row["ShiftEnd"]),
        "ContractHours": row["ContractHoursPerWeek"],
        "MaxDailyHours": row["MaxDailyHours"],
        "MaxWeeklyHours": row["MaxWeeklyHours"],
        "PreferredShift": row["PreferredShift"],
        "Status": row["Status"]
    })

# =========================
# CLEAN VISITS
# =========================
visits_clean = []

for _, row in visits.iterrows():
    visits_clean.append({
        "VisitID": row["VisitID"],
        "PatientID": row["PatientID"],
        "Day": row["Day"],
        "Latitude": row["Latitude"],
        "Longitude": row["Longitude"],
        "Duration": row["VisitDurationMinutes"],
        "TimeStart": time_to_minutes(row["TimeWindowStart"]),
        "TimeEnd": time_to_minutes(row["TimeWindowEnd"]),
        "Priority": priority_map[row["PriorityLevel"]],
        "Complexity": row["ComplexityLevel"],
        "Skills": to_set(row["RequiredSkills"]),
        "GenderPref": row["PreferredGender"],
        "LanguagePref": row["PreferredLanguage"]
    })

print("Carers cleaned:", len(carers_clean))
print("Visits cleaned:", len(visits_clean))

print("\nSample Carer:")
print(carers_clean[0])

print("\nSample Visit:")
print(visits_clean[0])

Carers cleaned: 462
Visits cleaned: 2207

Sample Carer:
{'CarerID': 'C0001', 'Date': '01/06/2026', 'Day': 'Monday', 'Latitude': 51.372966, 'Longitude': -0.357305, 'PreferredLat': 51.419129, 'PreferredLon': -0.332031, 'MaxDistance': 5, 'Experience': 1, 'Skills': {'Palliative Care', 'Medication Support'}, 'Gender': 'Other', 'Languages': {'French', 'English'}, 'Vehicle': 'Public Transport', 'WorkingDays': {'Monday', 'Friday', 'Tuesday', 'Thursday', 'Wednesday', 'Saturday'}, 'OffDays': {'Sunday'}, 'ShiftStart': 420, 'ShiftEnd': 780, 'ContractHours': 20, 'MaxDailyHours': 5, 'MaxWeeklyHours': 20, 'PreferredShift': 'Morning', 'Status': 'Available'}

Sample Visit:
{'VisitID': 'V000001', 'PatientID': 'P0001', 'Day': 'Tuesday', 'Latitude': 51.36008, 'Longitude': -0.273553, 'Duration': 45, 'TimeStart': 900, 'TimeEnd': 1080, 'Priority': 1, 'Complexity': 'Low', 'Skills': {'Personal Care', 'Companionship'}, 'GenderPref': 'No Preference', 'LanguagePref': 'Arabic'}


In [84]:
# =========================
# STEP 2: INDEXING
# =========================

# -------------------------
# VISIT INDEXING
# -------------------------
visit_id_to_idx = {}
idx_to_visit_id = {}

for i, v in enumerate(visits_clean):
    vid = v["VisitID"]
    visit_id_to_idx[vid] = i
    idx_to_visit_id[i] = vid

# -------------------------
# CARER-DAY INDEXING
# -------------------------
carer_id_to_idx = {}
idx_to_carer_id = {}

for i, c in enumerate(carers_clean):
    key = (c["CarerID"], c["Date"])
    carer_id_to_idx[key] = i
    idx_to_carer_id[i] = key

# -------------------------
# ADD INDEX FIELDS
# -------------------------
for v in visits_clean:
    v["idx"] = visit_id_to_idx[v["VisitID"]]

for c in carers_clean:
    key = (c["CarerID"], c["Date"])
    c["idx"] = carer_id_to_idx[key]

# -------------------------
# BUILD GA ARRAYS
# -------------------------
visits_array = visits_clean
carers_array = carers_clean

# -------------------------
# SUMMARY CHECK
# -------------------------
print("Total Visits Indexed:", len(visits_array))
print("Total Carer-Day Nodes Indexed:", len(carers_array))

print("\nSample Visit Indexed:")
print(visits_array[0])

print("\nSample Carer-Day Indexed:")
print(carers_array[0])

Total Visits Indexed: 2207
Total Carer-Day Nodes Indexed: 462

Sample Visit Indexed:
{'VisitID': 'V000001', 'PatientID': 'P0001', 'Day': 'Tuesday', 'Latitude': 51.36008, 'Longitude': -0.273553, 'Duration': 45, 'TimeStart': 900, 'TimeEnd': 1080, 'Priority': 1, 'Complexity': 'Low', 'Skills': {'Personal Care', 'Companionship'}, 'GenderPref': 'No Preference', 'LanguagePref': 'Arabic', 'idx': 0}

Sample Carer-Day Indexed:
{'CarerID': 'C0001', 'Date': '01/06/2026', 'Day': 'Monday', 'Latitude': 51.372966, 'Longitude': -0.357305, 'PreferredLat': 51.419129, 'PreferredLon': -0.332031, 'MaxDistance': 5, 'Experience': 1, 'Skills': {'Palliative Care', 'Medication Support'}, 'Gender': 'Other', 'Languages': {'French', 'English'}, 'Vehicle': 'Public Transport', 'WorkingDays': {'Monday', 'Friday', 'Tuesday', 'Thursday', 'Wednesday', 'Saturday'}, 'OffDays': {'Sunday'}, 'ShiftStart': 420, 'ShiftEnd': 780, 'ContractHours': 20, 'MaxDailyHours': 5, 'MaxWeeklyHours': 20, 'PreferredShift': 'Morning', 'Status'

In [85]:
import numpy as np

# =========================
# HAVERSINE FUNCTION
# =========================
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # miles

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


# =========================
# DISTANCE MATRIX
# =========================
num_carers = len(carers_array)
num_visits = len(visits_array)

distance_matrix = np.zeros((num_carers, num_visits))


for i, c in enumerate(carers_array):
    clat = c["Latitude"]
    clon = c["Longitude"]

    for j, v in enumerate(visits_array):
        vlat = v["Latitude"]
        vlon = v["Longitude"]

        distance_matrix[i][j] = haversine(clat, clon, vlat, vlon)

print("Distance matrix created:", distance_matrix.shape)

# =========================
# SAMPLE CHECK
# =========================
print("Distance Carer 0 → Visit 0:", distance_matrix[0][0])
print("Distance Carer 0 → Visit 1:", distance_matrix[0][1])

Distance matrix created: (462, 2207)
Distance Carer 0 → Visit 0: 3.7209783092698494
Distance Carer 0 → Visit 1: 25.15814799979125


In [86]:
#feasibility metrixs, (Strict Filtering) NOT Panelty based filtering
import numpy as np

num_carers = len(carers_array)
num_visits = len(visits_array)

feasible_matrix = np.zeros((num_carers, num_visits), dtype=int)


for i, c in enumerate(carers_array):
    for j, v in enumerate(visits_array):

        # -------------------------
        # HC1: Distance constraint
        # -------------------------
        if distance_matrix[i][j] > c["MaxDistance"]:
            continue

        # -------------------------
        # HC2: Shift constraint
        # -------------------------
        if v["TimeStart"] < c["ShiftStart"]:
            continue

        if v["TimeEnd"] > c["ShiftEnd"]:
            continue

        # -------------------------
        # HC3: Day compatibility
        # -------------------------
        if v["Day"] != c["Day"]:
            continue

        # -------------------------
        # HC4: Daily capacity (basic check)
        # -------------------------
        if v["Duration"] > c["MaxDailyHours"] * 60:
            continue

        # -------------------------
        # FEASIBLE
        # -------------------------
        feasible_matrix[i][j] = 1


print("Feasibility matrix created:", feasible_matrix.shape)
print("Feasible assignments:", np.sum(feasible_matrix))

# Sample check
print("Carer 0 feasible visits:", np.sum(feasible_matrix[0]))

Feasibility matrix created: (462, 2207)
Feasible assignments: 10575
Carer 0 feasible visits: 20


In [87]:
#candidate list construction
feasible_candidates = {}

num_visits = len(visits_array)
num_carers = len(carers_array)

for j in range(num_visits):
    feasible_candidates[j] = []

    for i in range(num_carers):
        if feasible_matrix[i][j] == 1:
            feasible_candidates[j].append(i)

print("Candidate lists created")
print("Example visit 0 candidates:", len(feasible_candidates[0]))

Candidate lists created
Example visit 0 candidates: 11


In [88]:
#initial population
import random

POP_SIZE = 30
NOISE_RATE = 0.1

def generate_individual():
    chromosome = []

    for j in range(num_visits):

        if len(feasible_candidates[j]) == 0:
            # fallback (rare case)
            chromosome.append(random.randint(0, num_carers-1))
            continue

        r = random.random()

        if r < (1 - NOISE_RATE):
            # FEASIBLE ASSIGNMENT
            chromosome.append(random.choice(feasible_candidates[j]))
        else:
            # NOISE (controlled infeasible)
            chromosome.append(random.randint(0, num_carers-1))

    return chromosome


# Generate population
population = [generate_individual() for _ in range(POP_SIZE)]

print("Initial population created:", len(population))
print("Chromosome length:", len(population[0]))

Initial population created: 30
Chromosome length: 2207


In [89]:
print("Population size:", len(population))
print("Sample chromosome:", population[0][:10])

Population size: 30
Sample chromosome: [290, 440, 345, 438, 317, 172, 172, 417, 171, 124]


In [90]:
#fitness function parameters
HARD_PENALTY_WEIGHT = 1000
SOFT_WEIGHT = 1

In [91]:
import numpy as np

def compute_fitness(chromosome):

    hard_violations = 0
    soft_score = 0

    workload = np.zeros(num_carers)

    total_hard_checks = 0

    for visit_idx, carer_idx in enumerate(chromosome):

        visit = visits_array[visit_idx]
        carer = carers_array[carer_idx]

        # -------------------------
        # HARD CONSTRAINT CHECKS
        # -------------------------

        total_hard_checks += 1

        # feasibility
        if feasible_matrix[carer_idx][visit_idx] == 0:
            hard_violations += 1

        # distance
        if distance_matrix[carer_idx][visit_idx] > carer["MaxDistance"]:
            hard_violations += 1

        # shift
        if visit["TimeStart"] < carer["ShiftStart"] or visit["TimeEnd"] > carer["ShiftEnd"]:
            hard_violations += 1

        # day mismatch
        if visit["Day"] != carer["Day"]:
            hard_violations += 1

        # -------------------------
        # SOFT CONSTRAINTS
        # -------------------------

        workload[carer_idx] += visit["Duration"]

        if len(visit["Skills"].intersection(carer["Skills"])) > 0:
            soft_score += 1

        if visit["LanguagePref"] in carer["Languages"]:
            soft_score += 1

        soft_score += visit["Priority"] * 0.2

    # -------------------------
    # NORMALIZED HARD SCORE
    # -------------------------
    hard_score = hard_violations / total_hard_checks

    # -------------------------
    # WORKLOAD FAIRNESS (NORMALIZED)
    # -------------------------
    avg_workload = np.mean(workload)
    workload_penalty = np.mean(np.abs(workload - avg_workload)) / (avg_workload + 1e-6)

    # -------------------------
    # FINAL FITNESS
    # -------------------------
    fitness = soft_score - (hard_score * 10) - (workload_penalty * 5)

    return fitness

In [92]:
print("Refined fitness:", compute_fitness(population[0]))
print("Refined fitness:", compute_fitness(population[1]))

Refined fitness: 2808.164693993451
Refined fitness: 2820.0212172268452


In [93]:
#tournament selection
import random

def tournament_selection(population, k=3):
    selected = random.sample(population, k)
    selected.sort(key=lambda ind: compute_fitness(ind), reverse=True)
    return selected[0]

In [94]:
#crossover uniform + safe design
def crossover(parent1, parent2):
    child1 = []
    child2 = []

    for i in range(len(parent1)):
        if random.random() < 0.5:
            child1.append(parent1[i])
            child2.append(parent2[i])
        else:
            child1.append(parent2[i])
            child2.append(parent1[i])

    return child1, child2

In [95]:
#repair function
def repair(chromosome):
    repaired = []

    for visit_idx, carer_idx in enumerate(chromosome):

        # if valid → keep
        if feasible_matrix[carer_idx][visit_idx] == 1:
            repaired.append(carer_idx)

        else:
            # fallback: choose feasible if available
            if len(feasible_candidates[visit_idx]) > 0:
                repaired.append(random.choice(feasible_candidates[visit_idx]))
            else:
                # worst-case fallback
                repaired.append(carer_idx)

    return repaired

In [96]:
#MUTATION (HYBRID FEASIBLE + RANDOM)
def mutate(chromosome, mutation_rate=0.1):

    mutated = chromosome.copy()

    for i in range(len(mutated)):

        if random.random() < mutation_rate:

            # 80% feasible mutation
            if random.random() < 0.8 and len(feasible_candidates[i]) > 0:
                mutated[i] = random.choice(feasible_candidates[i])

            # 20% exploration mutation (random)
            else:
                mutated[i] = random.randint(0, num_carers - 1)

    return mutated

In [97]:
#FULL GENERATION LOOP (CORE GA)
def run_ga_generation(population, elite_size=2):

    new_population = []

    # sort by fitness
    population = sorted(population, key=lambda x: compute_fitness(x), reverse=True)

    # ELITISM (keep best solutions)
    new_population.extend(population[:elite_size])

    while len(new_population) < len(population):

        # selection
        parent1 = tournament_selection(population, k=3)
        parent2 = tournament_selection(population, k=3)

        # crossover
        child1, child2 = crossover(parent1, parent2)

        # mutation
        child1 = mutate(child1)
        child2 = mutate(child2)

        # repair (VERY IMPORTANT)
        child1 = repair(child1)
        child2 = repair(child2)

        new_population.append(child1)

        if len(new_population) < len(population):
            new_population.append(child2)

    return new_population

In [98]:
new_pop = run_ga_generation(population)
print("New population size:", len(new_pop))
print("Best fitness:", max(compute_fitness(ind) for ind in new_pop))

New population size: 30
Best fitness: 2877.6187044163744


In [99]:
#dynamic event probability
EVENT_PROBABILITY = 0.2

In [100]:
#scenario generator
import random

def generate_scenario():

    event_type = random.choice([
        "new_visit",
        "carer_unavailable",
        "cancel_visit",
        "modify_time_window"
    ])

    return event_type

In [101]:
#applying scenario impact
def add_new_visit():
    new_visit = {
        "VisitID": f"NV{random.randint(1000,9999)}",
        "PatientID": "NEW",
        "Day": random.choice(["Monday","Tuesday","Wednesday","Thursday","Friday"]),
        "Latitude": random.uniform(51.3, 51.6),
        "Longitude": random.uniform(-0.4, 0.1),
        "Duration": 30,
        "TimeStart": 600,
        "TimeEnd": 1080,
        "Priority": 4,
        "Complexity": "High",
        "Skills": set(),
        "LanguagePref": "English",
        "GenderPref": "No Preference",
        "idx": len(visits_array)
    }

    visits_array.append(new_visit)

In [102]:
#carer unavailable
def deactivate_carer():
    idx = random.randint(0, len(carers_array)-1)
    for j in range(num_visits):
        feasible_matrix[idx][j] = 0
    return idx

In [103]:
#cancel visit
def cancel_visit():
    idx = random.randint(0, len(visits_array)-1)
    visits_array[idx]["cancelled"] = True
    return idx

In [104]:
#main dynamic GA Loop
GENERATIONS = 50
EVENT_PROBABILITY = 0.2

best_solution = None
best_fitness = float("-inf")

population_history = [] # Initialize population_history

for gen in range(GENERATIONS):

    print(f"Generation {gen}")

    # -------------------------
    # STEP 1: EVOLVE POPULATION
    # -------------------------
    population = run_ga_generation(population)
    population_history.append(population.copy()) # Store a copy of the population

    # -------------------------
    # STEP 2: TRACK BEST
    # -------------------------
    for ind in population:
        fit = compute_fitness(ind)
        if fit > best_fitness:
            best_fitness = fit
            best_solution = ind

    # -------------------------
    # STEP 3: DYNAMIC EVENT
    # -------------------------
    if random.random() < EVENT_PROBABILITY:

        event = generate_scenario()
        print("EVENT OCCURRED:", event)

        if event == "new_visit":
            add_new_visit()

        elif event == "carer_unavailable":
            deactivate_carer()

        elif event == "cancel_visit":
            cancel_visit()

        elif event == "modify_time_window":
            v = random.choice(visits_array)
            v["TimeEnd"] -= 60  # tighten window

        # -------------------------
        # STEP 4: REPAIR AFTER EVENT
        # -------------------------
        population = [repair(ind) for ind in population]

    print("Best fitness so far:", best_fitness)

Generation 0
Best fitness so far: 2870.8077241705278
Generation 1
Best fitness so far: 2874.6157409796233
Generation 2
Best fitness so far: 2880.6271128822823
Generation 3
Best fitness so far: 2902.6206565927528
Generation 4
Best fitness so far: 2902.6206565927528
Generation 5
Best fitness so far: 2922.7565973423452
Generation 6
EVENT OCCURRED: modify_time_window
Best fitness so far: 2922.7565973423452
Generation 7
Best fitness so far: 2931.804423295854
Generation 8
Best fitness so far: 2941.7887794574317
Generation 9
EVENT OCCURRED: cancel_visit
Best fitness so far: 2941.7887794574317
Generation 10
Best fitness so far: 2953.6347729684753
Generation 11
Best fitness so far: 2953.6347729684753
Generation 12
Best fitness so far: 2953.6347729684753
Generation 13
Best fitness so far: 2953.6347729684753
Generation 14
Best fitness so far: 2955.6695802360746
Generation 15
Best fitness so far: 2955.6695802360746
Generation 16
Best fitness so far: 2955.6695802360746
Generation 17
Best fitness so

In [105]:
best_solution   # final chromosome
visits_array
carers_array
distance_matrix
feasible_matrix

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [106]:
#performance evaluation
#computaional complexity
import time

start_time = time.time()

# run GA here
# run_ga()

end_time = time.time()

total_time = end_time - start_time

avg_time_per_generation = total_time / GENERATIONS

print("Total runtime:", total_time)
print("Avg time per generation:", avg_time_per_generation)

Total runtime: 2.4318695068359375e-05
Avg time per generation: 4.863739013671875e-07


In [107]:
#feasibility rate over time
def feasibility_rate(solution):
    valid = 0
    total = len(solution)

    for v_idx, c_idx in enumerate(solution):
        if feasible_matrix[c_idx][v_idx] == 1:
            valid += 1

    return valid / total

In [108]:
feasibility_over_time = []

for gen_pop in population_history:
    best = max(gen_pop, key=compute_fitness)
    feasibility_over_time.append(feasibility_rate(best))

In [109]:
#workload computation
import numpy as np

def compute_workload(solution):
    workload = np.zeros(len(carers_array))

    for v_idx, c_idx in enumerate(solution):
        workload[c_idx] += visits_array[v_idx]["Duration"]

    return workload

In [110]:
#workload variNCE
workload = compute_workload(best_solution)

workload_variance = np.var(workload)

print("Workload variance:", workload_variance)

Workload variance: 162873.7192190926


In [111]:
#WORKLOAD STANDARD DEVIATION
workload_std = np.std(workload)

print("Workload std deviation:", workload_std)

Workload std deviation: 403.5761628479717


In [112]:
#DISTANCE efficiency
def distance_efficiency(solution):
    total_distance = 0

    for v_idx, c_idx in enumerate(solution):
        total_distance += distance_matrix[c_idx][v_idx]

    return total_distance / len(solution)

In [113]:
#workload fairness index
def fairness_index(workload):
    workload = np.array(workload)
    mean = np.mean(workload)

    if mean == 0:
        return 1

    deviation = np.mean(np.abs(workload - mean))
    return 1 - (deviation / mean)

In [114]:
#coverage
def coverage(solution):
    assigned = 0

    for v_idx, c_idx in enumerate(solution):
        if c_idx is not None:
            assigned += 1

    return assigned / len(solution)

In [115]:
def evaluate_solution(solution):

    workload = compute_workload(solution)

    metrics = {
        "feasibility_rate": feasibility_rate(solution),
        "workload_variance": np.var(workload),
        "workload_std": np.std(workload),
        "distance_efficiency": distance_efficiency(solution),
        "fairness_index": fairness_index(workload),
        "coverage": coverage(solution)
    }

    return metrics

In [116]:
#full evaluuation
best_solution
visits_array
carers_array
distance_matrix

array([[ 3.72097831, 25.158148  , 25.158148  , ..., 20.05725595,
        20.05725595, 20.05725595],
       [ 3.72097831, 25.158148  , 25.158148  , ..., 20.05725595,
        20.05725595, 20.05725595],
       [ 3.72097831, 25.158148  , 25.158148  , ..., 20.05725595,
        20.05725595, 20.05725595],
       ...,
       [20.8366872 , 15.16728739, 15.16728739, ..., 22.04112992,
        22.04112992, 22.04112992],
       [20.8366872 , 15.16728739, 15.16728739, ..., 22.04112992,
        22.04112992, 22.04112992],
       [20.8366872 , 15.16728739, 15.16728739, ..., 22.04112992,
        22.04112992, 22.04112992]])

In [117]:
#metrics version
import numpy as np
import pandas as pd

def compute_all_metrics(solution):

    workload = np.zeros(len(carers_array))

    total_distance = 0
    valid = 0

    for v_idx, c_idx in enumerate(solution):

        # workload
        workload[c_idx] += visits_array[v_idx]["Duration"]

        # distance
        total_distance += distance_matrix[c_idx][v_idx]

        # feasibility
        if feasible_matrix[c_idx][v_idx] == 1:
            valid += 1

    # -------------------------
    # FEASIBILITY RATE
    # -------------------------
    feasibility_rate = valid / len(solution)

    # -------------------------
    # WORKLOAD METRICS
    # -------------------------
    workload_variance = np.var(workload)
    workload_std = np.std(workload)

    # fairness (Gini-like)
    mean_w = np.mean(workload)
    fairness_index = 1 - (np.mean(np.abs(workload - mean_w)) / (mean_w + 1e-6))

    # -------------------------
    # DISTANCE EFFICIENCY
    # -------------------------
    distance_efficiency = total_distance / len(solution)

    # -------------------------
    # COVERAGE
    # -------------------------
    coverage = len(solution) / len(visits_array)

    return {
        "feasibility_rate": feasibility_rate,
        "workload_variance": workload_variance,
        "workload_std": workload_std,
        "fairness_index": fairness_index,
        "distance_efficiency": distance_efficiency,
        "coverage": coverage
    }

In [118]:
metrics = compute_all_metrics(best_solution)

for k, v in metrics.items():
    print(k, ":", v)

feasibility_rate : 0.9610330765745355
workload_variance : 162873.7192190926
workload_std : 403.5761628479717
fairness_index : 0.2044687853290571
distance_efficiency : 6.787375158238002
coverage : 1.0


In [119]:
df_metrics = pd.DataFrame([metrics])
df_metrics.to_csv("ga_global_metrics.csv", index=False)

print("Saved: ga_global_metrics.csv")

Saved: ga_global_metrics.csv


In [121]:
import pandas as pd

metrics = compute_all_metrics(best_solution)

# print metrics (you already have this)
for k, v in metrics.items():
    print(k, ":", v)

# ⭐ SAVE TO CSV (GA RESULTS)
df = pd.DataFrame([metrics])

filename = "ga_final_results_instance2.csv"
df.to_csv(filename, index=False)

print(f"\n✅ GA results_Instance2_saved to: {filename}")

feasibility_rate : 0.9610330765745355
workload_variance : 162873.7192190926
workload_std : 403.5761628479717
fairness_index : 0.2044687853290571
distance_efficiency : 6.787375158238002
coverage : 1.0

✅ GA results_Instance2_saved to: ga_final_results_instance2.csv
